In [ ]:
# Install Tiny-SIMA and dependencies into the active Jupyter kernel.
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.60"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(
        "Tiny-SIMA requires a Python 3.11 or 3.12 Jupyter kernel."
    )

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

has_cuda = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    ).returncode == 0
)
extras = "notebook,cuda12" if has_cuda else "notebook"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])

# Editable installs add src/ through a .pth file, which is only processed
# when Python starts. Add it explicitly for this already-running kernel.
source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Repository:", repo_root)
print("Source path added to kernel:", source_root)
print("CUDA detected:", has_cuda)


# Phase 2: skill evaluation benchmark

This notebook validates the first action-grounding curriculum. Every episode uses real Craftax transitions but starts with one feasible nearby target or crafting setup.

The scripted policy should solve every episode of the first six skills. The later skills run on a longer horizon, and two of them stay stochastic for the scripted policy: a sapling drops one time in ten, and the cow can wander off the tile it was placed on. The random policy establishes a chance-level baseline before we train a goal-conditioned executor.

In [ ]:
import jax
import matplotlib.pyplot as plt

from tiny_sima.envs.craftax import render_state
from tiny_sima.envs.skills import make_skill_environment
from tiny_sima.evaluation.skills import evaluate_skill_suite
from tiny_sima.goals import ALL_SKILL_IDS, SKILL_IDS, SkillId

# The skills after the original six: the stone-pickaxe stage and the survival
# skills (stages A and B).
LATER_SKILL_IDS = ALL_SKILL_IDS[len(SKILL_IDS) :]

print("JAX devices:", jax.devices())
print("Skills:", [skill.value for skill in ALL_SKILL_IDS])


In [ ]:
environment = make_skill_environment(SkillId.COLLECT_WOOD, horizon=8)
_, episode_state = environment.reset(jax.random.PRNGKey(0))
pixels = render_state(
    episode_state.env_state,
    environment.bundle.spec.name,
    block_pixel_size=16,
)

plt.figure(figsize=(6, 6))
plt.imshow(pixels)
plt.axis("off")
plt.title("Nearby-target collect-wood scenario")
plt.show()


In [ ]:
NUM_EPISODES = 256
HORIZON = 8
# Sleeping from the scenario's energy of eight takes twelve steps, and a
# sapling or a cow can take several tries.
LATER_SKILL_HORIZON = 16
SEED = 0

results = [
    *evaluate_skill_suite(num_episodes=NUM_EPISODES, horizon=HORIZON, seed=SEED),
    *evaluate_skill_suite(
        num_episodes=NUM_EPISODES,
        horizon=LATER_SKILL_HORIZON,
        seed=SEED,
        skill_ids=LATER_SKILL_IDS,
    ),
]

for result in results:
    print(
        f"{result['skill_id']:<22} "
        f"{result['policy']:<8} "
        f"success={result['success_rate']:.3f} "
        f"length={result['mean_episode_length']:.2f}"
    )


In [ ]:
skill_names = [skill.value for skill in ALL_SKILL_IDS]
scripted = {
    result["skill_id"]: result["success_rate"]
    for result in results
    if result["policy"] == "scripted"
}
random = {
    result["skill_id"]: result["success_rate"]
    for result in results
    if result["policy"] == "random"
}
positions = list(range(len(skill_names)))
width = 0.38

plt.figure(figsize=(14, 4.5))
plt.bar(
    [position - width / 2 for position in positions],
    [scripted[name] for name in skill_names],
    width,
    label="Scripted",
)
plt.bar(
    [position + width / 2 for position in positions],
    [random[name] for name in skill_names],
    width,
    label="Random",
)
plt.xticks(positions, skill_names, rotation=30, ha="right")
plt.ylabel("Success rate")
plt.ylim(0, 1.05)
plt.title("Phase 2 nearby-target skill baselines")
plt.legend()
plt.tight_layout()
plt.show()
